# Block-CV R² Summary — gpt2-xl L36 worddur
Train-test (held-out) McFadden pseudo-R² across all patients, regions, and conditions.
All values are from 5-fold temporal block CV.

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

GLM_BASE = '/scratch/aniluchavez/ConvoDATAS/SemanticGLM'
SELFOTHER_DIR = f'{GLM_BASE}/gpt2-xl_ctx200_worddur/pc100'
ALLCOND_DIR   = f'{GLM_BASE}/gpt2-xl_ctx200_worddur_allcond/pc100'
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

def load_all(pkl_dir):
    rows = []
    for f in sorted(os.listdir(pkl_dir)):
        if not f.endswith('_sem.pkl'): continue
        obj = pickle.load(open(os.path.join(pkl_dir, f), 'rb'))
        df  = obj['df'] if isinstance(obj, dict) else obj
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

so  = load_all(SELFOTHER_DIR)
all_c = load_all(ALLCOND_DIR)

print('self/other:', so.shape,  '| patients:', so['patient'].nunique())
print('all_cond:  ', all_c.shape,'| patients:', all_c['patient'].nunique())
print('regions:', so['region'].unique())
print('conditions:', so['condition'].unique())

In [ ]:
# ── Summary table: % sig and median R² by region × condition ───────────────
def summary_table(df):
    rows = []
    for region in df['region'].unique():
        for cond in sorted(df['condition'].unique()):
            sub = df[(df['region']==region) & (df['condition']==cond)]
            sig = sub[sub['significant']]
            rows.append({
                'region': region, 'condition': cond,
                'n_neurons': len(sub),
                'n_sig': len(sig),
                'pct_sig': 100*len(sig)/len(sub) if len(sub) else np.nan,
                'median_r2_all': sub['r2'].median(),
                'median_r2_sig': sig['r2'].median() if len(sig) else np.nan,
            })
    return pd.DataFrame(rows)

print('=== self/other ===')
display(summary_table(so).round(4))
print('=== all_conditions ===')
display(summary_table(all_c).round(4))

In [ ]:
# ── PLOT 1: R² distributions by region × condition ─────────────────────────
REGIONS = so['region'].unique()
CONDS   = ['self', 'other']
COLORS  = {'self': '#2166ac', 'other': '#d6604d'}

fig, axes = plt.subplots(1, len(REGIONS), figsize=(5*len(REGIONS), 4.5), sharey=False)
if len(REGIONS) == 1: axes = [axes]

for ax, region in zip(axes, REGIONS):
    for cond in CONDS:
        sub = so[(so['region']==region) & (so['condition']==cond)]
        if sub.empty: continue
        ax.hist(sub['r2'], bins=40, alpha=0.5, color=COLORS[cond],
                label=f'{cond} (n={len(sub)}, {100*sub["significant"].mean():.0f}% sig)',
                density=True)
        med = sub['r2'].median()
        ax.axvline(med, color=COLORS[cond], lw=1.5, linestyle='--')
    ax.axvline(0, color='k', lw=0.8, linestyle=':')
    ax.set_xlabel('Test R² (McFadden)')
    ax.set_ylabel('Density')
    ax.set_title(region)
    ax.legend(fontsize=9)

plt.suptitle('gpt2-xl L36  |  5-fold temporal block CV  |  test R²', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_r2_dist_selfother.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 2: % significant per patient (dot plot) ───────────────────────────
REGIONS = so['region'].unique()
fig, axes = plt.subplots(1, len(REGIONS), figsize=(6*len(REGIONS), 4.5), sharey=True)
if len(REGIONS) == 1: axes = [axes]

for ax, region in zip(axes, REGIONS):
    pats = sorted(so['patient'].unique())
    xs   = np.arange(len(pats))
    for i, cond in enumerate(CONDS):
        pcts = []
        for pat in pats:
            sub = so[(so['patient']==pat) & (so['region']==region) & (so['condition']==cond)]
            pcts.append(100*sub['significant'].mean() if len(sub) else np.nan)
        offset = (i - 0.5) * 0.25
        ax.scatter(xs + offset, pcts, color=COLORS[cond], s=60, label=cond, zorder=3)
        ax.plot(xs + offset, pcts, color=COLORS[cond], alpha=0.3, lw=1)
    ax.axhline(5, color='gray', lw=0.8, linestyle='--', label='5% chance')
    ax.set_xticks(xs); ax.set_xticklabels([p.replace('_task','\n') for p in pats],
                                           fontsize=7, rotation=45, ha='right')
    ax.set_ylabel('% significant neurons'); ax.set_title(region)
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9)

plt.suptitle('% significant neurons per patient  |  FDR q<0.05', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_pct_sig_per_patient.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 3: Median R² of significant neurons per patient ───────────────────
fig, axes = plt.subplots(1, len(REGIONS), figsize=(6*len(REGIONS), 4.5), sharey=True)
if len(REGIONS) == 1: axes = [axes]

for ax, region in zip(axes, REGIONS):
    pats = sorted(so['patient'].unique())
    xs   = np.arange(len(pats))
    for i, cond in enumerate(CONDS):
        meds = []
        for pat in pats:
            sub = so[(so['patient']==pat) & (so['region']==region) &
                     (so['condition']==cond) & so['significant']]
            meds.append(sub['r2'].median() if len(sub) else np.nan)
        offset = (i - 0.5) * 0.25
        ax.scatter(xs + offset, meds, color=COLORS[cond], s=60, label=cond, zorder=3)
        ax.plot(xs + offset, meds, color=COLORS[cond], alpha=0.3, lw=1)
    ax.axhline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xticks(xs); ax.set_xticklabels([p.replace('_task','\n') for p in pats],
                                           fontsize=7, rotation=45, ha='right')
    ax.set_ylabel('Median test R² (sig neurons)'); ax.set_title(region)
    ax.legend(fontsize=9)

plt.suptitle('Median R² of significant neurons per patient', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/05_median_r2_per_patient.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 4: Self vs Other R² scatter (significant neurons only) ─────────────
for region in REGIONS:
    self_r2  = so[(so['region']==region) & (so['condition']=='self')  & so['significant']]['r2']
    other_r2 = so[(so['region']==region) & (so['condition']=='other') & so['significant']]['r2']
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Left: violin
    ax = axes[0]
    parts = ax.violinplot([self_r2.dropna(), other_r2.dropna()],
                          positions=[0,1], showmedians=True)
    for pc, c in zip(parts['bodies'], ['#2166ac','#d6604d']):
        pc.set_facecolor(c); pc.set_alpha(0.6)
    ax.set_xticks([0,1]); ax.set_xticklabels(['self','other'])
    ax.set_ylabel('Test R²'); ax.set_title(f'{region} — sig neurons')
    t, p = stats.mannwhitneyu(self_r2.dropna(), other_r2.dropna(), alternative='two-sided')
    ax.set_xlabel(f'Mann-Whitney p={p:.3f}')

    # Right: all_conditions R² distribution
    ax = axes[1]
    sub_all = all_c[(all_c['region']==region) & all_c['significant']]
    ax.hist(sub_all['r2'], bins=40, color='#4dac26', alpha=0.7,
            label=f'all_cond (n={len(sub_all)}, sig)', density=True)
    ax.axvline(sub_all['r2'].median(), color='#4dac26', lw=1.5, linestyle='--')
    ax.set_xlabel('Test R²'); ax.set_ylabel('Density')
    ax.set_title(f'{region} — all conditions (combined)')
    ax.legend(fontsize=9)

    plt.suptitle(f'{region}  |  gpt2-xl L36 block-CV', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/05_selfother_vs_allcond_{region}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── PLOT: Train vs Test R² — overfitting check (sig neurons only) ─────────
# Paired violins: train R² and test R² for each region × condition.
# Wilcoxon signed-rank tests whether gap (train−test) > 0.

if 'r2_train' not in so.columns:
    print('r2_train not in data — rerun after the updated semantic_glm.py completes')
else:
    REGIONS = list(so['region'].unique())
    CONDS   = ['self', 'other']
    COLORS  = {'self': '#2166ac', 'other': '#d6604d'}
    TRAIN_C = '#bdbdbd'

    fig, axes = plt.subplots(1, len(REGIONS),
                              figsize=(6 * len(REGIONS), 5), sharey=False)
    if len(REGIONS) == 1:
        axes = [axes]

    for ax, region in zip(axes, REGIONS):
        pos = 0
        tick_pos, tick_labs = [], []
        legend_handles = [mpatches.FancyArrow(0, 0, 0, 0, color=TRAIN_C, label='train')]
        legend_handles = []

        for cond in CONDS:
            sig_sub = so[(so['region']==region) & (so['condition']==cond) & so['significant']]\
                        .dropna(subset=['r2_train', 'r2'])
            if sig_sub.empty:
                pos += 3; continue

            tr = sig_sub['r2_train'].values
            te = sig_sub['r2'].values
            mtr, mte = np.median(tr), np.median(te)
            ratio = mte / mtr if mtr > 0 else np.nan
            _, p_wil = stats.wilcoxon(tr - te, alternative='greater')
            p_str = f'p={p_wil:.2g}' if p_wil >= 0.001 else f'p={p_wil:.1e}'

            for data, xpos, c in [(tr, pos, TRAIN_C), (te, pos+1, COLORS[cond])]:
                vp = ax.violinplot([data], positions=[xpos], widths=0.7, showmedians=True)
                for pc in vp['bodies']:
                    pc.set_facecolor(c); pc.set_alpha(0.65)
                vp['cmedians'].set_color('k'); vp['cmedians'].set_linewidth(1.5)
                for part in ['cbars', 'cmins', 'cmaxes']:
                    vp[part].set_color('k'); vp[part].set_linewidth(0.8)

            ax.plot([pos, pos+1], [mtr, mte], 'k-', lw=1.5, zorder=5)

            ymax = max(np.percentile(tr, 95), np.percentile(te, 95)) * 1.2
            ax.text((pos + pos+1) / 2, ymax,
                    f'ratio={ratio:.2f}\n{p_str}',
                    ha='center', va='bottom', fontsize=8)

            tick_pos  += [pos, pos+1]
            tick_labs += [f'train\n{cond}', f'test\n{cond}']
            pos += 3

        ax.axhline(0, color='gray', lw=0.8, linestyle='--')
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_labs, fontsize=9)
        ax.set_ylabel('McFadden R²')
        ax.set_title(region)

        # Build legend from violin body artists
        handles = [
            plt.matplotlib.patches.Patch(facecolor=TRAIN_C, alpha=0.65, label='train'),
        ] + [
            plt.matplotlib.patches.Patch(facecolor=COLORS[c], alpha=0.65, label=f'test ({c})')
            for c in CONDS
        ]
        ax.legend(handles=handles, fontsize=8, loc='upper right')

    plt.suptitle('Train vs Test R²  |  significant neurons only  |  gpt2-xl L36\n'
                 '5-fold temporal block CV  |  Wilcoxon: train > test?',
                 y=1.02)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/05_train_vs_test_r2.pdf', bbox_inches='tight')
    plt.show()

    print(f'\n{"region":<14} {"cond":<8} {"n_sig":>6} {"med_train":>10} {"med_test":>9} {"ratio":>7} {"Wilcoxon_p":>12}')
    for region in REGIONS:
        for cond in CONDS:
            sig_sub = so[(so['region']==region) & (so['condition']==cond) & so['significant']]\
                        .dropna(subset=['r2_train', 'r2'])
            if sig_sub.empty: continue
            tr, te = sig_sub['r2_train'].values, sig_sub['r2'].values
            mtr, mte = np.median(tr), np.median(te)
            ratio = mte / mtr if mtr > 0 else np.nan
            _, p = stats.wilcoxon(tr - te, alternative='greater')
            print(f'{region:<14} {cond:<8} {len(sig_sub):>6} {mtr:>10.4f} {mte:>9.4f} {ratio:>7.2f} {p:>12.3g}')